# V2 数据质量与 EDA

输入为 2026-10-20 12:00 声明快照。先在项目根目录运行 python -m experiments.v2_eda --run-id <新编号>，再把下方 RUN_ID 改成该编号。Notebook 只读取汇总表，不加载企业级原始记录；原始数据、来源 SQL 和全部统计口径以运行目录的 manifest 为准。U 是未标注，不是真负例。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RUN_ID = '20261022_v2_eda_1200_04'
ROOT = Path.cwd()
if not (ROOT / 'experiments' / 'v2_eda.py').exists():
    ROOT = ROOT.parent
RUN = ROOT / 'runs' / RUN_ID
assert (RUN / 'manifest.json').exists(), f'Missing run: {RUN}'
def load(name):
    return pd.read_csv(RUN / name, encoding='utf-8-sig')
print(RUN)


## 1. 规模、重复和已知 P 冲突

实体计数以规范化信用代码为键。缺代码行仅进入原始质量统计。P/U 交叉不代表 U 是负例。

In [ ]:
display(load('entity_quality.csv'))
display(load('entity_overlaps.csv'))
sizes = load('group_sizes.csv')
display(sizes)
ax = sizes.set_index('group')['entities'].plot.bar(figsize=(10, 4), color='#3572A5')
ax.set_ylabel('Distinct entities')
ax.set_xlabel('')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()


## 2. 缺失、非法值和异常

空值、占位文本和真实数字 0 分别处理。地区标记与地区文本不一致先视为待核查，不自动删除。

In [ ]:
missing = load('field_missingness_by_group.csv')
fields = ['establishment_date', 'registered_capital', 'paid_capital', 'insured_count', 'industry_major_category', 'company_size', 'has_company_introduction']
groups = ['P_SH_SZ', 'U1_nonP', 'P_national', 'U2_nonP', 'U3_nonP']
matrix = missing[missing.field.isin(fields) & missing.group.isin(groups)].pivot(index='field', columns='group', values='missing_rate').reindex(index=fields, columns=groups)
display((matrix * 100).round(1).rename_axis('Missing %'))
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(matrix.to_numpy() * 100, aspect='auto', cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(len(groups)), groups, rotation=30, ha='right')
ax.set_yticks(range(len(fields)), fields)
fig.colorbar(image, ax=ax, label='Missing %')
plt.tight_layout()
plt.show()
display(load('anomalies.csv').query('count > 0').sort_values(['cohort', 'count'], ascending=[True, False]).head(40))


## 3. P/U 分布比较

比较实体级代表记录；U 已排除全国 P 重合实体。P_SH_SZ 对 U1_nonP、P_national 对 U2/U3 才是主要比较。分布差异可能来自取数与标签选择机制，不能直接解释成保险福利效应。

In [ ]:
numeric = load('numeric_distribution.csv')
selected = numeric[numeric.group.isin(['P_SH_SZ', 'U1_nonP', 'P_national', 'U2_nonP', 'U3_nonP'])]
display(selected[['group', 'field', 'valid_n', 'median', 'q25', 'q75']].round(1))
insured = selected[selected.field.eq('insured_count_value')].set_index('group').reindex(['P_SH_SZ', 'U1_nonP', 'P_national', 'U2_nonP', 'U3_nonP'])
ax = insured['median'].plot.bar(figsize=(7, 4), color='#D18F3E')
ax.set_ylabel('Median insured count among nonmissing')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
category = load('categorical_distribution.csv')
display(category[category.field.eq('region_province') & category.group.isin(['P_national', 'U2_nonP', 'U3_nonP'])].head(30))


## 4. P 来源与标签质量候选

EDA 的 Gold 候选来源包括原力或量化Q3榜单；原始 p_quality 保留并单独报告差异。U 文件没有对等的来源平台字段，不能比较 P/U 平台来源占比。Gold 候选未经过独立业务核验。

In [ ]:
source = load('p_source_distribution.csv')
display(source[source.field.isin(['source_group', 'p_quality'])])
display(load('p_quality_candidate.csv'))
display(source[source.field.eq('source')].head(12))
display(load('p_missing_code_by_quality.csv'))
display(load('flag_rates_by_group.csv'))
display(load('field_missingness_raw.csv').query("cohort == 'P' and field in ['source', 'province_tag', 'city_tag']"))
